# 推测解码分析


In [ ]:
from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

os.environ["CUDA_VISIBLE_DEVICES"] = ""
_pkg_dir = Path(__file__).parent.resolve()
if str(_pkg_dir) not in sys.path:
    sys.path.insert(0, str(_pkg_dir))

_core_dir = _pkg_dir.parent / "core"
if str(_core_dir) not in sys.path:
    sys.path.insert(0, str(_core_dir))
from pipeline import Config, load_model_and_tokenizer

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

cfg = Config()
model, tokenizer, device = load_model_and_tokenizer(
    cfg.model_path, device=cfg.device,
    torch_dtype=getattr(torch, cfg.torch_dtype, torch.bfloat16),
)

prompt = "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n"
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32)
input_ids = inputs.input_ids.to(device)
T = input_ids.shape[1]
max_gen = 16

print(f"Prompt: {prompt}")
print(f"Tokens: {T}")

# 用主模型生成真实后续序列
gen_ids = input_ids.clone()
with torch.no_grad():
    for _ in range(max_gen):
        out = model(input_ids=gen_ids, use_cache=False, return_dict=True)
        next_tok = out.logits[0, -1].argmax().item()
        gen_ids = torch.cat([gen_ids, torch.tensor([[next_tok]], device=device)], dim=1)

print(f"Ground truth: {tokenizer.decode(gen_ids[0])}")
print()

# MTP 草稿：主模型生成 tok[T]，MTP 后续草稿 tok[T+1] 起
# MTP 在位置 T（通过 roll 看到 tok[T]）预测 tok[T+1]
draft = [gen_ids[0, T].item()]   # 主模型的第一个 token
cur_ids = gen_ids[:, :T+1].clone()   # 提示 + 第一个 token

with torch.no_grad():
    for step in range(max_gen - 1):
        # 获取 MTP 对下一个 token 的预测
        raw = model.model(input_ids=cur_ids, use_cache=False, return_dict=True)
        mtp_hidden = raw.mtp_hidden_states[0]   # [1, T+1, D]
        # MTP 在倒数第二个位置: the rolled input has the newest token here
        #（最后一个位置循环回到 tok0，跳过）
        mtp_logits = model.lm_head(mtp_hidden[0, -2:-1]).float()
        next_tok = mtp_logits[0].argmax().item()
        draft.append(next_tok)
        cur_ids = torch.cat([cur_ids, torch.tensor([[next_tok]], device=device)], dim=1)

draft_tokens = draft   # 长度 = max_gen（1 LM + 15 MTP）
print(f"\n{'='*80}")
print("MTP Speculative Draft (first token from LM, rest from MTP)")
print(f"{'='*80}")
print(f"{'pos':<5} {'source':<8} {'Draft':<20} {'Ground truth':<20} {'Match?'}")
print(f"{'-'*65}")

match_count = 0
for i in range(max_gen):
    src = "LM" if i == 0 else "MTP"
    dr = draft_tokens[i]
    truth = gen_ids[0, T + i].item()
    match = dr == truth
    if match: match_count += 1
    print(f"{i:<5} {src:<8} {tokenizer.decode(dr):<20} {tokenizer.decode(truth):<20} {'OK' if match else 'X'}")

print(f"\nDraft accuracy: {match_count}/{max_gen} ({match_count/max_gen*100:.1f}%)")

# 验证阶段：在完整草稿上运行主模型
full_ids = torch.cat([input_ids, torch.tensor([draft_tokens], device=device)], dim=1)
with torch.no_grad():
    verify_out = model(input_ids=full_ids, use_cache=False, return_dict=True)
verify_logits = verify_out.logits.float()

print(f"\n{'='*80}")
print("Speculative Decoding Verification")
print(f"{'='*80}")
print(f"\n{'pos':<5} {'Draft token':<20} {'LM predicts':<20} {'Accept?'}")
print(f"{'-'*65}")

accept_contiguous = 0
for i in range(max_gen):
    draft_tok = full_ids[0, T + i].item()
    lm_tok = verify_logits[0, T - 1 + i].argmax().item()
    accept = draft_tok == lm_tok
    if accept:
        accept_contiguous += 1
    else:
        break   # 推测解码：只接受连续前缀

    print(f"{i:<5} {tokenizer.decode(draft_tok):<20} {tokenizer.decode(lm_tok):<20} {'ACCEPT' if accept else 'REJECT'}")

if accept_contiguous < max_gen:
    rej_pos = accept_contiguous
    r_tok = full_ids[0, T + rej_pos].item()
    r_lm = verify_logits[0, T - 1 + rej_pos].argmax().item()
    print(f"{rej_pos:<5} {tokenizer.decode(r_tok):<20} {tokenizer.decode(r_lm):<20} {'REJECT (stop)'}")

print(f"\n=== Results ===")
print(f"Draft length: {max_gen} tokens")
print(f"Contiguously accepted: {accept_contiguous}/{max_gen}")
print(f"Effective tokens per 2 forward passes: {accept_contiguous + 1}")
if accept_contiguous > 0:
    print(f"Speedup vs sequential: ~{(accept_contiguous+1)/2:.1f}x (1 draft + 1 verify = 2 passes)")
else:
    print(f"No contiguous acceptance - MTP draft doesn't match decoder's predictions at position 0")

# 也检查：如果用 MTP hidden -> gate 做路由 during verification?
print(f"\n{'='*80}")
print("Routing prediction during verification pass")
print(f"{'='*80}")
print(f"Using MTP hidden state at position t to predict decoder routing at t+1")
print(f"(This is what would enable expert prefetching during verification)")
print(f"")
print(f"Our earlier tests (predict_test.py) showed this works with Cos=0.81 avg.")
print(f"During verification, ALL N positions are processed simultaneously.")
print(f"→ MTP hidden at each position predicts routing for ALL layers at that position")
print(f"→ All experts can be batch-loaded before verification starts")

print("\nDone.")

---
## sd_multi（跨提示对比）


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

cfg = Config()
model, tokenizer, device = load_model_and_tokenizer(
    cfg.model_path, device=cfg.device,
    torch_dtype=getattr(torch, cfg.torch_dtype, torch.bfloat16),
)

prompts = {
    "code_fib": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n",
    "english": "The quick brown fox jumps over the lazy dog, but the dog was",
    "code_sort": "def merge_sort(arr):\n    if len(arr) <= 1:\n        return arr\n    mid = len(arr)",
}
max_gen = 4

print(f"\n{'='*80}")
print("Spec Decoding: MTP draft across prompt types")
print(f"{'='*80}")
print(f"{'Prompt':<20} {'Toks':<6} {'Draft/GT':<14} {'Accept':<10} {'Spd':<6} {'Draft tokens accepted'}")
print(f"{'------':<20} {'----':<6} {'--------':<14} {'------':<10} {'---':<6} {'---'}")

for pname, prompt in prompts.items():
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32)
    input_ids = inputs.input_ids.to(device)
    T = input_ids.shape[1]

    # 真实值
    gen_ids = input_ids.clone()
    with torch.no_grad():
        for _ in range(max_gen):
            out = model(input_ids=gen_ids, use_cache=False, return_dict=True)
            next_tok = out.logits[0, -1].argmax().item()
            gen_ids = torch.cat([gen_ids, torch.tensor([[next_tok]], device=device)], dim=1)

    # MTP 草稿：LM 第一个 token，然后 MTP 自回归
    draft = [gen_ids[0, T].item()]
    cur_ids = gen_ids[:, :T+1].clone()
    with torch.no_grad():
        for step in range(max_gen - 1):
            raw = model.model(input_ids=cur_ids, use_cache=False, return_dict=True)
            mtp = raw.mtp_hidden_states[0]
            logits = model.lm_head(mtp[0, -2:-1]).float()
            ntok = logits[0].argmax().item()
            draft.append(ntok)
            cur_ids = torch.cat([cur_ids, torch.tensor([[ntok]], device=device)], dim=1)

    # 验证
    full = torch.cat([input_ids, torch.tensor([draft], device=device)], dim=1)
    with torch.no_grad():
        vout = model(input_ids=full, use_cache=False, return_dict=True)
    vlogits = vout.logits.float()

    accept = 0
    for i in range(max_gen):
        if full[0, T+i].item() == vlogits[0, T-1+i].argmax().item():
            accept += 1
        else:
            break

    dm = sum(1 for i in range(max_gen) if draft[i] == gen_ids[0, T+i].item())
    sp = (accept + 1) / 2

    # 显示接受的 token
    acc_toks = []
    for i in range(max_gen):
        dt = full[0, T+i].item()
        vt = vlogits[0, T-1+i].argmax().item()
        if dt == vt:
            acc_toks.append(tokenizer.decode(dt).strip() or '\\n')
        else:
            break

    acc_str = ' '.join(acc_toks) if acc_toks else '(none)'
    print(f"{pname:<20} {T:<6} {dm}/{max_gen} ({dm/max_gen*100:.0f}%){accept:<5}/{max_gen}{sp:<6.1f}x {acc_str[:50]}")

print(f"\nDone.")

---
## verify_spec（搬运量模拟）


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

cfg = Config()
model, tokenizer, device = load_model_and_tokenizer(
    cfg.model_path, device=cfg.device,
    torch_dtype=getattr(torch, cfg.torch_dtype, torch.bfloat16),
)

prompt = "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)"
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=64)
input_ids = inputs.input_ids.to(device)
T = input_ids.shape[1]
tokens = [tokenizer.decode(t) for t in input_ids[0]]

print(f"\nPrompt: {prompt}")
print(f"Tokens: {T}")
print()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        output_router_logits=True,
        use_cache=False,
        return_dict=True,
    )

all_router = outputs.router_logits
num_layers = len(all_router) - 1
K = cfg.num_experts_per_tok

# 每层每位置的专家
# all_router[l][0]: [1, T, E]
layer_experts = {}
for l in range(num_layers):
    router = all_router[l][0].squeeze(0)
    topk = router.topk(K, dim=-1).indices
    layer_experts[l] = topk    # [T, K]

模拟推测解码：一次验证 d 个草稿 token

顺序基线：每次生成 1 个 token

In [ ]:

print("=" * 70)
print("Simulation: Sequential vs Speculative Decoding")
print("=" * 70)

# 非推测基线: generate N tokens one at a time
# 每步生成处理 1 个 token → 1 set of experts per layer
# 跨 N 步，可能多次加载相同的专家 (redundant)

# 推测模式：一次验证 N 个草稿 token
# N 个位置同时处理 → expert sets can be shared

for N in [2, 4, 8, 16]:
    if N >= T:
        continue
    
    print(f"\n--- N={N} draft tokens ---")
    
    layers_all = range(num_layers)
    for layer_type, layer_range in [("All layers", layers_all),
                                      ("Shallow (0-4)", range(min(5, num_layers))),
                                      ("Middle (5-14)", range(5, min(15, num_layers))),
                                      ("Deep (15-19)", range(15, num_layers))]:
        
        # 基线顺序：N 个位置加载的唯一专家, one at a time
        seq_unique = []
        for l in layer_range:
            all_seen = set()
            for t in range(N):
                all_seen.update(layer_experts[l][t].tolist())
            seq_unique.append(len(all_seen))
        
        # 推测模式：一次通过需要的唯一专家数
        #（某些位置可能共享专家）
        spec_unique = []
        for l in layer_range:
            experts_at_all_pos = set()
            for t in range(N):
                experts_at_all_pos.update(layer_experts[l][t].tolist())
            spec_unique.append(len(experts_at_all_pos))
        
        avg_seq = sum(seq_unique) / len(seq_unique)
        avg_spec = sum(spec_unique) / len(spec_unique)
        
        # 节省：顺序模式需要加载-卸载-再加载
        # 推测模式：一次加载并共享
        # 总专家加载次数：
        # 顺序：每步每层加载 K 个专家 → N × len(layers) × K
        # 推测：每层只加载一次唯一专家 → sum(unique_per_layer)
        seq_total_loads = N * len(layer_range) * K
        spec_total_loads = sum(spec_unique)
        savings = (1 - spec_total_loads / seq_total_loads) * 100
        
        print(f"  {layer_type:<20} sequential: {seq_total_loads:4d} loads, "
              f"speculative: {spec_total_loads:3d} loads, "
              f"savings: {savings:5.1f}%")


关键指标：当一次验证 N 个草稿 token 时，

每层有多少唯一的专家被激活？

In [ ]:
print("\n" + "=" * 70)
print("Unique experts per layer during N-token verification pass")
print("=" * 70)

print(f"\n{'Layer':<8} {'N=2':<10} {'N=4':<10} {'N=8':<10} {'N=16':<10}")
print(f"{'-----':<8} {'----':<10} {'----':<10} {'----':<10} {'-----':<10}")
for l in range(min(20, T)):
    row = [f"L{l:<4}"]
    for N in [2, 4, 8, 16]:
        if N < T:
            experts = set()
            for t in range(N):
                experts.update(layer_experts[l][t].tolist())
            row.append(f"{len(experts):<6}/8{'':>4}")
        else:
            row.append(f"{'N/A':<10}")
    print("  ".join(row))

# 所有层的平均值
print(f"\n{'Avg':<8}", end="")
for N in [2, 4, 8, 16]:
    if N < T:
        totals = []
        for l in range(20):
            experts = set()
            for t in range(N):
                experts.update(layer_experts[l][t].tolist())
            totals.append(len(experts))
        avg = sum(totals) / len(totals)
        print(f" {avg:.1f}/8{'':>8}", end="")
print()


顺序生成：每 token 唯一专家数（最坏情况）

In [ ]:
print("\n" + "=" * 70)
print("Sequential generation: unique expert count per layer across N tokens")
print("(Each token loads K new experts; no sharing)")
print("=" * 70)

print(f"\n{'Layer':<8} {'N=1':<10} {'N=2':<10} {'N=4':<10} {'N=8':<10} {'N=16':<10}")
print(f"{'-----':<8} {'----':<10} {'----':<10} {'----':<10} {'----':<10} {'-----':<10}")
for l in range(min(20, T)):
    row = [f"L{l:<4}"]
    for N in [1, 2, 4, 8, 16]:
        if N < T:
            row.append(f"{min(N*K, 256):<6}{'':>4}")
        else:
            row.append(f"{'N/A':<10}")
    print("  ".join(row))

print("\nDone.")
